In [2]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

def render_as_image(env):
    plt.imshow(env.render())
    plt.axis('off')
    plt.show()


In [5]:
import numpy as np
import gymnasium as gym

ENV_ID = 'CliffWalking-v1' if 'CliffWalking-v1' in gym.registry else 'CliffWalking-v0'
print('Using', ENV_ID, '| gymnasium', gym.__version__)


np.random.seed(42)

def render_as_image(env):
    """Show the current frame of the environment as a single image"""
    plt.figure(figsize=(6, 2))
    plt.imshow(env.render())
    plt.axis('off')
    plt.show()


Using CliffWalking-v1 | gymnasium 1.4.0


### Q-table | Agent
* `Qtable` – Q(s, a) stored as a 48 × 4 numpy array.
* `chooseActionEpsilonGreedyPolicy` – random action with probability ε, otherwise the best one.
* `playQ_learning` – off-policy TD control: target `r + γ·max_a' Q(s', a')`.
* `play_SARSA` – on-policy TD control: target `r + γ·Q(s', a')`, where `a'` is the action really taken next.

In [12]:
DEFAULT_QVALUE = 0
NEGATIVE_INF = -100

class Qtable:

    def __init__(self, environment = None):
        self.n_states = environment.observation_space.n  ## 48
        self.n_actions = environment.action_space.n     ## 4
        # Q(s, a) = 0 for all pairs at the start (also for the terminal state,
        # which must stay 0 - it is never updated, because we never act from it)
        self.q = np.full((self.n_states, self.n_actions), DEFAULT_QVALUE, dtype=float)

    def set_qvalue(self, state, action, value):
        self.q[state, action] = value

    def get_qvalue(self, state, action):
        return self.q[state, action]

    # max_a Q(s, a)  -> used by Q-learning as the "best possible future"
    def get_max_qvalue(self, state):
        return np.max(self.q[state])

    #get action values  as vector indexed by env action space
    def get_qvalue_vector(self, state):
        return self.q[state].copy()

    # get state values as maximum from state-action values originating from state
    # V(s) = max_a Q(s, a)  -> dict {state_id: value}, used by showValues()
    def get_state_values(self):
        return {s: float(np.max(self.q[s])) for s in range(self.n_states)}


class Agent:

    def __init__(self):
        self.current_state = None
        self.environment = gym.make(ENV_ID)

        self.lr = 0.01            # alpha - learning rate (step size)
        self.exp_rate = 0.3       # epsilon - probability of a random (exploratory) action
        self.decay_gamma = 1.0    # gamma - discount factor

        # initial state reward
        self.state_values = {}
        self.state_visit_count = {}

        # initial Q values
        self.Q_values = Qtable(self.environment)

        # sum of rewards of every training episode (for learning curves)
        self.episode_rewards = []

    def end_training(self):
        self.environment.close()

    # =====================================================================
    # Q-LEARNING  (off-policy TD control)
    #   Q(s,a) <- Q(s,a) + alpha * [ r + gamma * max_a' Q(s',a') - Q(s,a) ]
    # =====================================================================
    def playQ_learning(self,  number_of_episodes=1):

        episode_count = 0

        while episode_count < number_of_episodes:
            i = 0
            terminated = False

            self.reset()
            R = 0
            while terminated == False:
                state = self.current_state

                # 1.BEHAVIOUR policy: epsilon-greedy (explores)
                action = self.chooseActionEpsilonGreedyPolicy(state, self.exp_rate)

                # 2. Act in the environment, observe reward and next state
                next_state, reward, terminated, truncated, info = self.environment.step(action)

                # 3. TD target uses the BEST action in s' (greedy TARGET policy),
                #    no matter which action the agent will really take next -> off-policy.
                #    In a terminal state there is no future, so the target is just r.                
                if terminated:
                    td_target = reward
                else:
                    td_target = reward + self.decay_gamma * self.Q_values.get_max_qvalue(next_state)
                
                # 4. Move Q(s,a) a step of size alpha towards the target
                old_q = self.Q_values.get_qvalue(state, action)
                td_error = td_target - old_q
                self.Q_values.set_qvalue(state, action, old_q + self.lr * td_error)

                # 5. Go to the next state
                R += reward
                i += 1
                self.current_state = next_state
                if truncated:          # time limit (not used by CliffWalking, but safe)
                    break

            self.episode_rewards.append(R)
            episode_count += 1

        self.state_values = self.Q_values.get_state_values()
        self.end_training()



    # =====================================================================
    # SARSA  (on-policy TD control)
    #   Q(s,a) <- Q(s,a) + alpha * [ r + gamma * Q(s',a') - Q(s,a) ]
    #   name = the 5 values used in one update: S, A, R, S', A'
    # =====================================================================
    def play_SARSA(self,  number_of_episodes=1):

        episode_count = 0


        while episode_count < number_of_episodes:
            i = 0
            terminated = False

            self.reset()
            R = 0
            # SARSA needs the first action BEFORE the loop
            action = self.chooseActionEpsilonGreedyPolicy(self.current_state, self.exp_rate)

            while terminated == False:
                state = self.current_state

                # 1. Act, observe reward and next state
                next_state, reward, terminated, truncated, info = self.environment.step(action)

                # 2. Choose the NEXT action with the SAME epsilon-greedy policy
                #    (this action will really be executed in the next step)
                next_action = self.chooseActionEpsilonGreedyPolicy(next_state, self.exp_rate)

                # 3. TD target uses Q of the action we will REALLY take -> on-policy.
                #    Random steps towards the cliff lower Q here, so SARSA learns
                #    to keep a safe distance from the cliff.
                if terminated:
                    td_target = reward
                else:
                    td_target = reward + self.decay_gamma * self.Q_values.get_qvalue(next_state, next_action)

                # 4. Update Q(s,a)
                old_q = self.Q_values.get_qvalue(state, action)
                td_error = td_target - old_q
                self.Q_values.set_qvalue(state, action, old_q + self.lr * td_error)

                # 5. S <- S',  A <- A'
                R += reward
                i += 1
                self.current_state = next_state
                action = next_action
                if truncated:
                    break

            self.episode_rewards.append(R)
            episode_count += 1


        self.state_values = self.Q_values.get_state_values()
        self.end_training()


    def follow_best_policy(self, show_images=True, max_steps=100):
        """
        Run ONE episode with the greedy policy (epsilon = 0) and show every step
        as an image. Frames are also stored in self.frames (used for the video).
        max_steps protects against an endless loop if the policy is not learned yet.
        """
        self.environment = gym.make(ENV_ID, render_mode='rgb_array')

        self.current_state = self.environment.reset()[0]
        self.frames = [self.environment.render()]
        self.path = [self.current_state]
        i = 0
        R = 0.0
        terminated = False
        while terminated == False and i < max_steps:
            if show_images:
                render_as_image(self.environment)
                print('State: ', self.current_state, '  ', self.Q_values.get_qvalue_vector(self.current_state))

            action = self.chooseActionEpsilonGreedyPolicy(self.current_state, 0.0)  # epsilon = 0 => strictly greedy policy
            next_state, reward, terminated, truncated, info = self.environment.step(action)
            if show_images:
                print(f'Step {i}: observation={next_state}, reward={reward}, done={terminated}')

            R = R + reward
            i = i + 1
            self.current_state = next_state
            self.frames.append(self.environment.render())
            self.path.append(next_state)

        if show_images:
            render_as_image(self.environment)      # final frame (agent in the goal)
        print("Reward ", R, "| steps", i)
        self.environment.close()
        return R

    def chooseActionEpsilonGreedyPolicy(self, state, epsilon):
        """
        Epsilon-greedy:
            with probability epsilon     -> random action    (exploration)
            with probability 1 - epsilon -> best known action (exploitation)
        Ties between equally good actions are broken randomly
        (np.argmax would always pick the first one, e.g. always 'up' at the start).
        """
        if np.random.random() < epsilon:
            action = np.random.randint(self.Q_values.n_actions)
        else:
            q = self.Q_values.get_qvalue_vector(state)
            best_actions = np.flatnonzero(q == q.max())
            action = int(np.random.choice(best_actions))

        return action
    
    def reset(self):
        self.states_visited_trace = []
        self.states_reward_trace = []
        self.current_state = self.environment.reset()[0]

    def showValues(self):
        BOARD_ROWS = 4
        BOARD_COLS = 12

        print('---------STATE VALUES------------')
        for i in range(0, BOARD_ROWS):
            print('----------------------------------')
            out = '| '
            for j in range(0, BOARD_COLS):
                state_id = i * 12 + j
                if (state_id) in self.state_values.keys():
                    value = self.state_values[state_id]
                else:
                    value = 0.0
                out += "{0:0.2f}".format(value).ljust(6) + ' | '
            print(out)
        print('----------------------------------')
    

agent = Agent()
agent.lr = 0.5
agent.play_SARSA(500)
agent.showValues()



---------STATE VALUES------------
----------------------------------
| -29.21 | -28.32 | -26.81 | -26.41 | -22.95 | -22.37 | -14.86 | -11.75 | -11.95 | -7.65  | -10.19 | -3.70  | 
----------------------------------
| -29.63 | -27.09 | -27.75 | -25.04 | -23.11 | -22.69 | -23.72 | -18.94 | -12.51 | -6.34  | -4.09  | -2.10  | 
----------------------------------
| -30.62 | -31.10 | -28.84 | -28.32 | -29.92 | -22.69 | -27.92 | -22.58 | -17.45 | -14.41 | -13.41 | -1.00  | 
----------------------------------
| -31.95 | 0.00   | 0.00   | 0.00   | 0.00   | 0.00   | 0.00   | 0.00   | 0.00   | 0.00   | 0.00   | 0.00   | 
----------------------------------


In [7]:
agent.follow_best_policy()

NameError: name 'agent' is not defined